# In-Context Learning and Prompt Templates with OpenAI

This is the OpenAI version of the IBM lab, built on the ports-and-adapters structure in this folder.

| Layer | Folder | What lives there |
|---|---|---|
| Domain | `domain/` | `GenerationSettings` |
| Application | `application/` | `LLMPort`, `RunTemplate`, `prompt_library` |
| Adapters | `adapters/` | `OpenAILLM`, `FakeLLM` |
| Driving adapter / composition root | this notebook | wires an adapter into the use cases |

**Differences from the IBM notebook**
- `WatsonxLLM` is replaced by `OpenAILLM` (LangChain's `ChatOpenAI` behind our `LLMPort`).
- `max_new_tokens` is `max_tokens`. `top_k` does not exist in OpenAI's API, so it is dropped.
- No `LLMChain`, no `ibm-watsonx-ai`, no kernel restart cell.

**Before you run:** the root `.env` must contain `OPENAI_API_KEY=...` and the kernel must be the project's `.venv`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

load_dotenv()

import os

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
from adapters.fake_llm import FakeLLM
from adapters.openai_llm import OpenAILLM
from application import prompt_library as prompts
from application.use_cases import RunTemplate
from domain.settings import GenerationSettings
from langchain_core.prompts import PromptTemplate

### Wire the adapter (composition root)

This is the only cell that names a provider. To run the whole notebook offline for free, swap in `FakeLLM()` instead.

`ask()` is a small notebook helper so each experiment reads like the original lab: pass a prompt and a few settings.

In [ ]:
llm = OpenAILLM(model="gpt-4o-mini")
# llm = FakeLLM("fake reply")   # offline, free


def ask(prompt: str, **params) -> str:
    """Send a prompt through the port with optional setting overrides."""
    return llm.generate(prompt, GenerationSettings(**params))


def show(prompt: str, **params) -> None:
    print(f"prompt: {prompt}\n")
    print(f"response: {ask(prompt, **params)}\n")

In [ ]:
show("Say hello in five words.", max_tokens=20)

## Prompt engineering

**In-context learning** is a model picking up a task from the prompt alone, with no change to its weights.

### Basic prompt

A basic prompt is just the start of some text. Note that `gpt-4o-mini` is a *chat* model, so it may reply rather than purely continue.

In [ ]:
show("The wind is ", max_tokens=128, temperature=0.5)

Try changing `max_tokens` and `temperature` and re-running.

### Exercise 1

Compare the three prompts. (Reference solution, rewrite it in your own words.)

In [ ]:
for p in [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The benefits of sustainable energy include",
]:
    show(p, max_tokens=128, temperature=0.5)

### Zero-shot prompt

An instruction with no examples.

In [ ]:
prompt = """Classify the following statement as true or false:
'The Eiffel Tower is located in Berlin.'

Answer:"""
show(prompt, temperature=0.0, max_tokens=100)

### Exercise 2

Three zero-shot prompts. (Reference solution.)

In [ ]:
movie_review_prompt = """Classify the following movie review as either 'positive' or 'negative'.

Review: "I was extremely disappointed by this film. The plot was predictable and the acting was wooden."

Classification:"""

climate_change_prompt = """Summarize the following paragraph about climate change in no more than two sentences.

Paragraph: "Climate change refers to long-term shifts in temperatures and weather patterns. Since the 1800s, human activities have been the main driver, primarily through burning fossil fuels, which produces heat-trapping gases. Consequences include more severe droughts, storms and heat waves, rising sea levels and melting glaciers."

Summary:"""

translation_prompt = """Translate the following English phrase into Spanish.

English: "I would like to order a coffee with milk and two sugars, please."

Spanish:"""

responses = {
    "movie_review": ask(movie_review_prompt, temperature=0.0, max_tokens=20),
    "climate_change": ask(climate_change_prompt, temperature=0.2, max_tokens=120),
    "translation": ask(translation_prompt, temperature=0.0, max_tokens=60),
}

for name, response in responses.items():
    print(f"=== {name.upper()} ===")
    print(response)
    print()

### One-shot prompt

One worked example shows the model the pattern and the format.

In [ ]:
prompt = """Here is an example of translating a sentence from English to French:

English: "How is the weather today?"
French: "Comment est le temps aujourd'hui?"

Now, translate the following sentence from English to French:

English: "Where is the nearest supermarket?"
French:"""
show(prompt, temperature=0.1, max_tokens=40)

### Exercise 3

Three one-shot prompts. (Reference solution.)

In [ ]:
formal_email_prompt = """Here is an example of a formal email requesting information:

Subject: Inquiry Regarding Product Specifications for Model XYZ-100

Dear Customer Support Team,

I am writing to request detailed specifications for your product Model XYZ-100, including its dimensions, power requirements and compatibility with accessories. Could you please provide this at your earliest convenience?

Sincerely,
John Smith

---

Now write a formal email to a university admissions office asking about the application deadline and required documents for the Master's program in Computer Science."""

technical_concept_prompt = """Here is an example of explaining a technical concept simply:

Technical Concept: Blockchain
Simple Explanation: A blockchain is like a digital notebook that many people hold copies of. When someone adds an entry, everyone's copy updates, and nothing already written can be erased.

---

Now explain this concept the same way:

Technical Concept: Machine Learning
Simple Explanation:"""

keyword_extraction_prompt = """Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now extract keywords from this sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation and reducing chemical inputs."
Keywords:"""

responses = {
    "formal_email": ask(formal_email_prompt, temperature=0.3, max_tokens=300),
    "technical_concept": ask(technical_concept_prompt, temperature=0.3, max_tokens=150),
    "keyword_extraction": ask(keyword_extraction_prompt, temperature=0.0, max_tokens=60),
}

for name, response in responses.items():
    print(f"=== {name.upper()} ===")
    print(response)
    print()

### Few-shot prompt

Several examples establish the pattern. Low `max_tokens` keeps the answer to a single label.

In [ ]:
prompt = """Here are a few examples of classifying emotions in statements:

Statement: 'I just won my first marathon!'
Emotion: Joy

Statement: 'I can't believe I lost my keys again.'
Emotion: Frustration

Statement: 'My best friend is moving to another country.'
Emotion: Sadness

Now, classify the emotion in the following statement:
Statement: 'That movie was so scary I had to cover my eyes.'
Emotion:"""
show(prompt, temperature=0.0, max_tokens=10)

### Chain-of-thought (CoT) prompt

Asking for intermediate steps improves multi-step reasoning, because each written token becomes input for the next.

In [ ]:
prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. How many apples are there now?'

Break down each step of your calculation."""
show(prompt, temperature=0.5, max_tokens=512)

### Exercise 4

Two CoT prompts. (Reference solution.)

In [ ]:
decision_making_prompt = """A student is deciding whether to study tonight or go to a movie with friends. They have a test in two days.

Think through this decision step by step, weighing the pros and cons of each option and which factors matter most."""

sandwich_making_prompt = """Explain how to make a peanut butter and jelly sandwich.

Break down each step of the process, from gathering ingredients to finishing the sandwich."""

responses = {
    "decision_making": ask(decision_making_prompt, temperature=0.5, max_tokens=512),
    "sandwich_making": ask(sandwich_making_prompt, temperature=0.5, max_tokens=512),
}

for name, response in responses.items():
    print(f"=== {name.upper()} ===")
    print(response)
    print()

### Self-consistency

Sample several independent solutions and keep the answer they agree on. This is the lab's single-prompt version: one call that asks for three independent calculations.

A true implementation calls the model N times at a higher temperature, extracts each `Final answer:` line and takes the majority vote. That is the `SelfConsistency` use case you are building by hand.

In [ ]:
prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

Provide three independent calculations and explanations, then determine the most consistent result."""
show(prompt, temperature=0.7, max_tokens=512)

## Applications with prompt templates

A `PromptTemplate` is a reusable prompt with named `{variables}`. Print the formatted prompt first, so you can see exactly what the model will receive.

In [ ]:
template = PromptTemplate.from_template("Tell me a {adjective} joke about {content}.")
print(template.format(adjective="funny", content="chickens"))

`RunTemplate` is our use case. It fills the template, sends it through the `LLMPort`, and returns the text. Under the hood it is a LangChain `template | RunnableLambda(...)` chain.

In [ ]:
joke = RunTemplate(llm, template, GenerationSettings(temperature=0.8, max_tokens=100))
print(joke.execute(adjective="funny", content="chickens"))
print()
print(joke.execute(adjective="sad", content="fish"))

### Text summarization

In [ ]:
content = """
The rapid advancement of technology in the 21st century has transformed healthcare, education, and transportation.
Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems.
AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are reducing traffic congestion.
Online learning platforms are making education more accessible around the world.
These developments enhance productivity and contribute to a more interconnected and informed society.
"""

summarize = RunTemplate(llm, prompts.SUMMARIZE, GenerationSettings(temperature=0.2, max_tokens=100))
print(summarize.execute(content=content))

### Question answering

The template tells the model to answer "Unsure about answer" rather than guess.

In [ ]:
context = """
The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets.
The inner planets (Mercury, Venus, Earth and Mars) are rocky and solid.
The outer planets (Jupiter, Saturn, Uranus and Neptune) are much larger and gaseous.
"""

qa = RunTemplate(llm, prompts.ANSWER, GenerationSettings(temperature=0.0, max_tokens=100))
print(qa.execute(question="Which planets in the solar system are rocky and solid?", content=context))
print(qa.execute(question="How many moons does Jupiter have?", content=context))

### Text classification

In [ ]:
classify = RunTemplate(llm, prompts.CLASSIFY, GenerationSettings(temperature=0.0, max_tokens=20))
print(
    classify.execute(
        text="The concert last night was an exhilarating experience with outstanding performances by all artists.",
        categories="Entertainment, Food and Dining, Technology, Literature, Music.",
    )
)

### Code generation (SQL)

This template is defined here in the notebook. Once you like it, move it into `application/prompt_library.py`.

In [ ]:
SQL = PromptTemplate.from_template(
    "Generate an SQL query based on the {description}\n\nSQL Query:"
)

description = """
Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days.
The table 'purchases' contains a column 'purchase_date'.
"""

sql_generation = RunTemplate(llm, SQL, GenerationSettings(temperature=0.0, max_tokens=200))
print(sql_generation.execute(description=description))

### Role playing

`role`, `tone` and `question` are separate variables, so you switch behaviour by changing one value.

`chat()` runs a `while` loop with `input()`. It is wrapped in a function so **Run All does not block**. Call it when you want to talk. Type `quit`, `exit` or `bye` to leave.

In [ ]:
ROLE = PromptTemplate.from_template(
    "You are an expert {role}. I have this question: {question}. "
    "I would like our conversation to be {tone}.\n\nAnswer:"
)

role = "Dungeons & Dragons game master"
tone = "engaging and immersive"

roleplay = RunTemplate(llm, ROLE, GenerationSettings(temperature=0.8, max_tokens=300))


def chat():
    while True:
        query = input("Question: ")
        if query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break
        print("Answer:", roleplay.execute(role=role, question=query, tone=tone))


# One non-interactive turn so you can see it work:
print(roleplay.execute(role=role, question="Who are you?", tone=tone))

In [ ]:
# chat()   # uncomment to start the interactive loop

### Exercise 5: product review analyzer

Sentiment, mentioned features and a one-sentence summary, using the `REVIEW` template from the prompt library. (Reference solution.)

Next step for you: replace the free-text reply with a typed `ReviewAnalysis` domain object and an `AnalyzeReview` use case.

In [ ]:
review_analysis = RunTemplate(llm, prompts.REVIEW, GenerationSettings(temperature=0.2, max_tokens=512))

reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful.",
]

for i, review in enumerate(reviews, start=1):
    print(f"==== Review #{i} ====")
    print(review_analysis.execute(review=review))
    print()

## Conclusion

You moved from basic prompts through zero-shot, one-shot, few-shot, chain-of-thought and self-consistency, then wrapped templates in a use case.

The provider appears in exactly one line (`llm = OpenAILLM(...)`). Swap it for `FakeLLM()` or a future adapter and nothing else changes.

**Still to build by hand**
- `SelfConsistency` with `Final answer:` extraction and a vote
- `Sentiment` and `ReviewAnalysis` in `domain/`, plus an `AnalyzeReview` use case
- Moving `SQL` and `ROLE` into `prompt_library.py`